# Inspect a saved MuRIL run — no GPU needed

Choose **Runtime → Change runtime type → CPU**, then **Run all** and authorize Drive. This reads the active v2 run and downloads a small `muril-recovery.zip` to share for review. It does not train, install ML dependencies, deserialize models, or delete anything. It contains metrics, configuration and file sizes, but no checkpoint weights or training passages. Keep the active v2 folder intact.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import subprocess
import sys
import json

drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/AI-Detector/full-muril-gpu-v2-scipy114')
REPO = Path('/content/AI-Detector-recovery')
SOURCE_COMMIT = 'b9cf2495bfccf3008676ce052f6539b313328d59'
def run(args):
    subprocess.run([str(x) for x in args], check=True)
if not REPO.exists():
    run(['git', 'clone', '--no-checkout', 'https://github.com/parthpathak719/AI-Detector.git', REPO])
run(['git', '-C', REPO, 'fetch', 'origin', SOURCE_COMMIT])
run(['git', '-C', REPO, 'checkout', '--detach', SOURCE_COMMIT])
sys.path.insert(0, str(REPO))
from scripts.inspect_full_run import inspect_run
OUTPUT = Path('/content/muril-recovery.zip')
summary = inspect_run(RUN_ROOT, OUTPUT)
print('Completed fine-tuned folds:', summary['completed_finetuned_folds'], '/', summary['planned_finetuned_folds'])
print(json.dumps(summary['tasks'], indent=2))
print('Warnings:', summary['warnings'])
print('Active saved checkpoint size (GiB):', round(sum(p['bytes'] for p in summary['checkpoints']) / 2**30, 2))
print('This does not include Drive Bin storage. No Drive files were changed or deleted.')
files.download(str(OUTPUT))


## Resume later

When GPU access is available, open the updated [training notebook](https://colab.research.google.com/github/parthpathak719/AI-Detector/blob/main/notebooks/Full_MuRIL_Finetuning.ipynb). Keep `full-muril-gpu-v2-scipy114` as the run folder. Completed folds are reused. Local epoch snapshots prevent repeated multi-GB Drive replacements; a runtime reset may repeat the unfinished fold. Do not run an old notebook copy: its cells still use the earlier checkpoint writer.

Review the Drive Bin separately. Delete only verified obsolete snapshots after checking that the active run still contains the checkpoint you need. Clearing storage does not reset Colab GPU quota.
